# CLIP_Smoothing_Teacher_Student_UDA_Seed42

CLIP B/16을 기본으로 **동일 조건 Aug / Teacher–Student UDA / 스무딩 대응 Teacher–Student UDA**를 독립 학습하고 CDF 공식 test에서 평가한다. GitHub에 새 코드를 push한 뒤 실행한다. 새 ZIP/TAR는 만들지 않는다.

|조건|FF++|CDF|
|---|---|---|
|Matched Aug|원본+무작위 약/중/강, 실제 정답 CE|사용하지 않음|
|Teacher–Student UDA|같은 source CE|EMA 교사의 신뢰도 높은 임시 정답으로 학생 학습, 두 view는 서로 반대 좌우반전|
|Teacher–Student UDA + smoothing|같은 source CE|동일한 관측 CDF에 학생 측 추가 약한 스무딩을 적용하여 임시 정답 학습|

- 시작점은 기존 FF++ c23 baseline best.pt. 작은 점수 보정기를 붙이지 않는다. CLIP 마지막 baseline-configured 2개 block·ln_post·visual projection·classifier를 학습하고 나머지는 고정한다. MODELS를 바꾸면 기존 Xception 전체/DINO 마지막 블록 범위도 지원한다. 기존 split·native ROI·전처리·스무딩 구현·fallback은 그대로다.
- FF++ 6,000장/epoch(진짜3,000 + 네 가짜 기법 각750), source 원본+랜덤 스무딩 두 view 평균CE. 세 전략은 source 목록·순서·view seed·초기값을 맞춘다. Source bilateral 결과를 캐시한다.
- CDF는 기존 dataset_inputs의 공식 test 영상 제외 및 보수적인 filename parent 연결 제외를 그대로 적용하고, 비테스트 5,000장을 정답을 버린 뒤 영상 균형으로 고정 선택한다. **실제 정답으로 5:5를 맞추지 않는다.** 이후 epoch마다 영상 균등 복원 추출6,000회. 영상별 균등이며 실제 세계 클래스 비율을 재현한다고 주장하지 않는다. 기존 GT-5:5 실험과는 타깃 샘플링이 다르므로 직접적인 방법 효과 비교로 쓰지 않는다.
- CDF 관측 입력은 원본/약/중/강 각각25% 선택. 교사에게 깨끗한 대응 원본·강도·정답을 주지 않는다. 학생의 추가 약한 스무딩은 이미 관측된 이미지에 재적용하므로 강도0.25+0.75를 단순 합산한 필터가 아니다. fallback은 변형하지 않는다.
- 교사는 학생의 EMA(decay0.999), gradient 없음/eval/FP32. 학생은 BF16 지원시 BF16, 아니면 FP32. 신뢰도>=0.9만 hard pseudo-label로 학습한다. target CE는 선택 표본 합을 **전체 target 배치 크기**로 나누며, 최대weight0.5를2epoch에 걸쳐 선형 증가한다. 신뢰도는 정답 보장이 아니며 채택률·진짜/가짜 임시 정답 수·전체 교사 예측 수를 epoch마다 기록한다. 비율을 강제로50:50으로 만들거나 테스트 정답을 확인해 임계값을 바꾸지 않는다.
- 최대10epoch, patience3/min_delta0.0001, batch16 source pairs, AdamW encoder/head LR1e-5/weight decay1e-4/gradient norm1. BN 통계는 고정. **같은 FF++ validation1,200장×4조건 평균 BER 최소**로 학생 best.pt를 선택한다. epoch0 후보이고 AUC guard는 없다. 교사의 검증 결과로 별도 선택하지 않는다. 학습 loss는 classification이며 AUC를 직접 최적화하는 학습이 아니다.
- 현재 Aug를 새로 학습하는 이유는 동일 검증 선택과 source batch/view 조건을 맞추기 위해서다. 과거 clean AUC로 선택한 Aug는 동일 대조군으로 대체하지 않는다.
- UDA는 source12,000views + target teacher/student 각6,000forward/epoch라 Aug보다 계산량이 크다. frozen feature 캐시는 사용할 수 없다. 비싼 bilateral은 source/target별 native PNG로 캐시하며 후보의 추가 smoothing도 별도로 캐시한다. 첫 epoch는 느릴 수 있다. 진행률은 직접 !python -u로 표시한다.
- 검증만 source1200×4, 테스트는 기존 CDF518영상/8,288장×4조건. FP32 frame FPR/FNR/BER, raw margin AUC 및 확률 AUC를 저장한다. test/target 정답으로 학습·모델 선택하지 않는다. 이 CDF test는 반복 확인한 개발 평가이며 최종 독립 holdout은 아니다. 인물 중복 허용, 확인되지 않은 source/driver 관계 완전 분리 주장 없음.
- 테스트에서는 **학생 모델 1회 forward**만 필요하다. 교사나 원본 대응짝은 필요 없다. Ranking 실험의 동일 checkpoint·manifest·스무딩·FP32 raw-margin baseline 예측은 있으면 재사용하고, 없으면 baseline만 다시 평가한다. 새 모델은3개×4조건. epoch0이 best면 결과가 baseline과 같을 수 있다; 테스트를 보고 last로 교체하지 않는다.
- 저장: comparison.csv, condition_means.csv, mixture_summary.csv(네 품질을 동일 비율로 섞은 AUC), cross_condition_auc.csv(4×4 교차 품질 AUC), 조건별 JSON/예측CSV(raw_margin 포함), config/history/best/last/completed, CDF 비라벨 목록, epoch별 target exposure plan. 같은 이미지의 네 view는 독립 표본이 아니다. run별 완료 skip, 마지막 완료 epoch 모델·EMA·optimizer 복원. epoch 도중 중단하면 그 epoch 처음부터 재개한다. 같은 run을 두 런타임에서 동시에 쓰지 않는다.


## Cell 1 — 설치·코드 갱신

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json, shutil

PROJECT=Path('/content/instagram_deepfake_robustness')
if PROJECT.exists():
    subprocess.run(['git','-C',str(PROJECT),'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/uyh1556/instagram-deepfake-robustness.git',str(PROJECT)],check=True)
assert (PROJECT/'scripts/ffpp_c23_smoothing_mean_teacher.py').is_file(),'새 코드를 GitHub에 push하세요.'
subprocess.run([sys.executable,'-m','pip','install','-q','setuptools==80.9.0','wheel'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-build-isolation','-r',str(PROJECT/'requirements-ffpp-c23-smoothing-eval.txt')],check=True)
subprocess.run([sys.executable,'-c','import torch; assert torch.cuda.is_available(),"GPU 런타임 선택"; print(torch.cuda.get_device_name(0))'],check=True)


## Cell 2 — 모델·경로

In [ ]:
MODELS=['clip_b16']  # 다른 모델: 'xception', 'dinov2_b14'
STRATEGIES=['augmentation','mean_teacher','mean_teacher_smoothing']
ROOT=Path('/content/drive/MyDrive/instagram-deepfake-robustness')
FF_DATA=ROOT/'data/ffpp_c23_4methods_roi16_v1'
CDF_DATA=ROOT/'data/celebdf_v2_face_crops16_v1'
SOURCE_ROOT=Path('/content/ffpp_c23_native_roi16_v1')
TARGET_ROOT=Path('/content/celebdf_v2_face_crops16_v1')
BASELINE_DRIVE_RUNS=ROOT/'runs/ffpp_c23_4methods_baseline_v1'
BASELINE_RUNS=Path('/content/ffpp_c23_baseline_eval_runs')
EXPERIMENT='ffpp_c23_smoothing_mean_teacher_v1'
GROUP='_'.join(MODELS)
RUNS_ROOT=ROOT/'runs'/EXPERIMENT
MANIFEST_ROOT=ROOT/'manifests'/EXPERIMENT/'seed42'/GROUP
EVALUATION_ROOT=ROOT/'evaluations'/EXPERIMENT/'seed42'/GROUP
VALIDATION_CACHE=Path('/content/smoothing_mean_teacher_source_validation_v1')
TRAIN_CACHE=Path('/content/smoothing_pivot_bilateral_cache_v1')
TARGET_TRAIN_CACHE=Path('/content/smoothing_mean_teacher_target_cache_v1')
CDF_CACHE=Path('/content/celebdf_official_smoothing_cache_v1')
CDF_PIXEL_CACHE=Path('/content/smoothing_ranking_cdf_bilateral_cache_v1')
RANKING_EVALUATIONS=ROOT/'evaluations/ffpp_c23_smoothing_ranking_v1/seed42'
MODEL_ARGS=' '.join(MODELS)
STRATEGY_ARGS=' '.join(STRATEGIES)
COMMON=f'--models {MODEL_ARGS} --strategies {STRATEGY_ARGS} --seed 42 --source-root "{SOURCE_ROOT}" --target-root "{TARGET_ROOT}" --baseline-runs-root "{BASELINE_RUNS}" --runs-root "{RUNS_ROOT}" --manifest-root "{MANIFEST_ROOT}" --output-root "{EVALUATION_ROOT}" --validation-cache "{VALIDATION_CACHE}" --train-cache "{TRAIN_CACHE}" --target-train-cache "{TARGET_TRAIN_CACHE}" --cache-root "{CDF_CACHE}" --evaluation-pixel-cache "{CDF_PIXEL_CACHE}" --reference-evaluations "{RANKING_EVALUATIONS}" --workers 2'
print('Models:',MODELS,'| Strategies:',STRATEGIES,flush=True)
print('Runs:',RUNS_ROOT,flush=True)


## Cell 3 — FF++ Train/Val·초기 모델

In [ ]:
!python -u "{PROJECT}/scripts/prepare_ffpp_c23_training_data.py" --drive-data "{FF_DATA}" --data-root "{SOURCE_ROOT}" --trainval-only
assert _exit_code==0,'FF++ 준비 실패: 위 오류 확인'

for model in MODELS:
    name=f'{model}_ffpp_c23_roi16_baseline_v1_seed42'
    source=BASELINE_DRIVE_RUNS/name
    dest=BASELINE_RUNS/name
    dest.mkdir(parents=True,exist_ok=True)
    for filename in ['best.pt','config.json','completed.json']:
        src,dst=source/filename,dest/filename
        assert src.is_file(),src
        if dst.is_file() and dst.stat().st_size==src.stat().st_size:
            continue
        print('Copying:',model,filename,flush=True)
        temp=dst.with_suffix(dst.suffix+'.copying')
        shutil.copy2(src,temp)
        assert temp.stat().st_size==src.stat().st_size,'복사 미완료'
        temp.replace(dst)


## Cell 4 — CDF 데이터 준비

In [ ]:
archives={
    'celebdf_v2_roi16_test_v1.tar':557015040,
    'celebdf_v2_roi16_non_test_real_v1.tar':826142720,
    'celebdf_v2_roi16_non_test_fake_v1.tar':5679564800,
}
for name,size in archives.items():
    manifest=TARGET_ROOT/'manifests'/f'{name.removesuffix(".tar")}.csv'
    markers=[Path('/content')/f'.{p}_{name}.done' for p in ['adapt','sda','sda_balance','mixed_uda','consistency_c','response','pivot','ranking','mean_teacher']]
    if name=='celebdf_v2_roi16_test_v1.tar':
        markers += [Path('/content/.celebdf_official_roi16_test_v1.done'),Path('/content/.preservation_b_celebdf_official_test.done')]
    if manifest.is_file() and any(p.exists() for p in markers):
        print('Already extracted:',name,flush=True)
        continue
    source=CDF_DATA/name
    assert source.is_file() and source.stat().st_size==size,f'TAR 업로드 확인: {source}'
    assert shutil.disk_usage('/content').free>2*size+2*2**30,'Colab 로컬 공간 부족'
    local=Path('/content')/name
    print('Copying:',name,flush=True)
    shutil.copy2(source,local)
    assert local.stat().st_size==size,'복사 미완료'
    print('Extracting:',name,flush=True)
    result=subprocess.run(['tar','-xf',str(local),'-C','/content'],capture_output=True,text=True)
    if result.returncode:
        raise RuntimeError(result.stderr[-5000:])
    assert manifest.is_file(),manifest
    (Path('/content')/f'.mean_teacher_{name}.done').touch()
    local.unlink()


## Cell 5 — 세 조건 학습

In [ ]:
!python -u "{PROJECT}/scripts/ffpp_c23_smoothing_mean_teacher.py" train {COMMON} --epochs 10 --patience 3 --source-images 6000 --val-images 1200 --target-images 5000 --batch-size 16 --encoder-lr 1e-5 --head-lr 1e-5 --pseudo-threshold 0.9 --target-weight 0.5 --ramp-epochs 2 --ema-decay 0.999
assert _exit_code==0,'학습 실패: 위 오류 확인 후 Cell 5 재실행'


## Cell 6 — CDF 원본·스무딩 평가

In [ ]:
!python -u "{PROJECT}/scripts/ffpp_c23_smoothing_mean_teacher.py" evaluate {COMMON} --batch-size 32
assert _exit_code==0,'평가 실패: 위 오류 확인 후 Cell 6 재실행'


## Cell 7 — 결과·임시 정답 분포

In [ ]:
import csv
from IPython.display import Markdown,display
labels={'baseline':'Baseline','augmentation':'Matched Aug','mean_teacher':'Teacher–Student UDA','mean_teacher_smoothing':'Teacher–Student UDA + smoothing'}
conditions={'normal':'원본','smooth_weak':'약','smooth_medium':'중','smooth_strong':'강'}
rows=list(csv.DictReader((EVALUATION_ROOT/'comparison.csv').open()))
means=list(csv.DictReader((EVALUATION_ROOT/'condition_means.csv').open()))
for model in MODELS:
    table=[f'### {model} — 조건별 결과','','|방법|조건|FPR|FNR|BER|AUC|','|---|---|---:|---:|---:|---:|']
    for r in rows:
        if r['model']==model:
            table.append(f"|{labels[r['strategy']]}|{conditions[r['condition']]}|{float(r['real_fpr']):.2%}|{float(r['fake_fnr']):.2%}|{float(r['balanced_error_rate']):.2%}|{float(r['auc']):.4f}|")
    display(Markdown('\n'.join(table)))
    table=['### 원본·약·중·강 평균','','|방법|FPR|FNR|BER|AUC|','|---|---:|---:|---:|---:|']
    for r in means:
        if r['model']==model:
            table.append(f"|{labels[r['strategy']]}|{float(r['real_fpr']):.2%}|{float(r['fake_fnr']):.2%}|{float(r['balanced_error_rate']):.2%}|{float(r['auc']):.4f}|")
    display(Markdown('\n'.join(table)))
    for strategy in STRATEGIES:
        run=RUNS_ROOT/f'{model}_{strategy}_{EXPERIMENT}_seed42'
        done=json.loads((run/'completed.json').read_text())
        print(strategy,'selected epoch:',done['selected_epoch'],'/ trained:',done['epochs_completed'])
        history=json.loads((run/'history.json').read_text()) if (run/'history.json').is_file() else []
        if strategy!='augmentation':
            table=['','|Epoch|임시 정답 채택률|진짜로 채택|가짜로 채택|Val BER|Val AUC|','|---:|---:|---:|---:|---:|---:|']
            for h in history:
                p=h['pseudo_selected_counts']
                table.append(f"|{h['epoch']}|{h['pseudo_selected_fraction']:.1%}|{p['real']}|{p['fake']}|{h['validation']['mean_ber']:.2%}|{h['validation']['mean_auc']:.4f}|")
            display(Markdown('\n'.join(table)))
print('CDF 정답은 샘플링/학습/체크포인트 선택에 사용하지 않습니다. AUC는 raw margin 기준입니다.')
print('평가 모델은 학생의 best.pt이며, epoch0이면 추가 학습 전 베이스라인이 선택된 것입니다.')
print('결과:',EVALUATION_ROOT)
